# Imports and file reading.

In [33]:
import pandas as pd
import glob
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial import cKDTree

In [34]:
JOINED_GLOB = "data/alphaearth_wetland_joined/*.parquet"
JOINED_FILES = sorted(glob.glob(JOINED_GLOB))

# Soft ceiling on the training pool. 10M rows lands at about 92% of all
# label==1 pixels statewide under a pure global sort, a reasonable place to
# start; raise it if the model wants more negatives to learn from.
SOFT_ROW_CAP = 10_000_000

# Geographic strata for the budget split. K=10 gives roughly 170 blocks per
# stratum on average
N_STRATA = 10
SEED = 0

# Training block pool: maximize label==1 captured, whole blocks, soft row cap

Pick whole 10km blocks from `data/alphaearth_wetland_joined/` for the training pool. The
objective is to capture as many label==1 (converted to developed) rows as possible without
the row count running far past a soft cap. Sorting all 1,727 blocks by positives per row and
taking the top of the list does this but clumps the picks into whichever corridor is
converting fastest right now, so instead the row budget gets split across geographic strata
first, each stratum sized by its share of statewide positives, and the density sort runs
within each stratum. That keeps the pool concentrated on real signal without it collapsing
onto one region.

## Step 1: block summary

One pass over `block_id` and `label` only, no embedding columns, so this stays fast.

In [35]:
# Loop over the blocks, record their density which is positive labels over total number of data rows in that block. 
block_parts = []
for f in JOINED_FILES:
    part = pd.read_parquet(f, columns=["block_id", "label"])
    part["is_pos"] = (part["label"] == 1).astype(np.int32)
    g = part.groupby("block_id").agg(n_rows=("is_pos", "size"), n_label1=("is_pos", "sum"))
    block_parts.append(g.reset_index())

block_summary = pd.concat(block_parts, ignore_index=True)
# block_id is unique to one tile file, so nothing here needs a second groupby.
assert block_summary["block_id"].is_unique

block_summary["density"] = block_summary["n_label1"] / block_summary["n_rows"]
block_summary

,block_id,n_rows,n_label1,density
0,b0239_0321,742,0,0.000000
1,b0239_0322,2999,0,0.000000
2,b0239_0323,4255,0,0.000000
3,b0239_0324,9815,10,0.001019
4,b0239_0325,20303,0,0.000000
...,...,...,...,...
1722,b0301_0391,15159,13,0.000858
1723,b0301_0392,1457,1,0.000686
1724,b0301_0393,2994,0,0.000000
1725,b0302_0390,2468,1,0.000405


## Step 2: geographic position per block

`block_id` already encodes the 10km block grid position stamped during the join, e.g.
`b0239_0323` becomes `brow` 239, `bcol` 323. Pull it back out rather than recomputing from
`x`/`y`.

In [36]:
block_summary[["brow", "bcol"]] = (
    block_summary["block_id"].str.extract(r"b(\d+)_(\d+)").astype(int)
)

## Step 3: geographic strata

KMeans over `(brow, bcol)` rather than a rectangular grid cut. Florida's coastline and
panhandle leave most of the bounding box empty, only 1,727 of the 5,670 possible grid cells
are populated, so a fixed grid would leave some strata with almost no blocks in them.

In [37]:
# Group the coordinates of all of the blocks into roughly 10 clusters. 
coords = block_summary[["brow", "bcol"]].to_numpy()
km = KMeans(n_clusters=N_STRATA, random_state=SEED, n_init=10)
block_summary["stratum"] = km.fit_predict(coords)

block_summary.groupby("stratum").agg(
    n_blocks=("block_id", "size"), n_label1=("n_label1", "sum"), n_rows=("n_rows", "sum")
)

,n_blocks,n_label1,n_rows
stratum,,,
0,161,25718,4493951
1,187,939,6068203
2,157,4101,8377360
3,138,5412,2620129
4,176,28359,6436897
5,159,10907,4029281
6,212,15494,5728714
7,209,7009,7958584
8,163,50423,6367113


## Step 4: row budget per stratum

Each stratum gets a slice of the soft cap proportional to its share of statewide positives.
A stratum with a lot of conversion gets a bigger budget, but capped at its own share, so no
single stratum can consume the whole pool the way a global density sort does.

In [38]:
# stratum budget is stratum positives / total dataset positives * the soft row cap. 
total_pos = block_summary["n_label1"].sum()
stratum_pos = block_summary.groupby("stratum")["n_label1"].sum()
stratum_budget = (stratum_pos / total_pos * SOFT_ROW_CAP).round().astype(int)
stratum_budget

stratum
0    1550136
1      56598
2     247185
3     326205
4    1709321
5     657413
6     933891
7     422463
8    3039215
9    1057574
Name: n_label1, dtype: int64

## Step 5: greedy select within each stratum

Highest positives per row first, take blocks until the running row total would cross that
stratum's budget. `cum_before < budget` always keeps the first block in a stratum (its rows
taken so far are zero) and lets the one block that actually crosses the line in too, since
the cap is soft; a stratum with zero budget naturally gets nothing. Ties break on `block_id`
so a rerun always picks the identical set.

In [39]:
selected_parts = []

# Get all of the blocks, in descendingd density. 
for stratum, budget in stratum_budget.items():
    sub = block_summary[block_summary["stratum"] == stratum].sort_values(
        ["density", "block_id"], ascending=[False, True]
    )
    # Creates an array that is a running total of all the blocks positives
    cum_before = sub["n_rows"].cumsum() - sub["n_rows"]
    # Boolean array of how many blocks to keep 
    keep = cum_before < budget
    selected_parts.append(sub[keep]) # Append the blocks who are true

selected_blocks = pd.concat(selected_parts, ignore_index=True)

print(f"blocks selected: {len(selected_blocks)} of {len(block_summary)}")
print(f"rows: {selected_blocks['n_rows'].sum():,} (soft cap {SOFT_ROW_CAP:,})")
print(
    f"label==1 captured: {selected_blocks['n_label1'].sum():,} of {total_pos:,} "
    f"({selected_blocks['n_label1'].sum() / total_pos:.1%})"
)
selected_blocks

blocks selected: 447 of 1727
rows: 10,233,887 (soft cap 10,000,000)
label==1 captured: 151,864 of 165,908 (91.5%)


,block_id,n_rows,n_label1,density,brow,bcol,stratum
0,b0273_0374,4710,640,0.135881,273,374,0
1,b0267_0375,8093,613,0.075744,267,375,0
2,b0272_0374,11781,832,0.070622,272,374,0
3,b0267_0374,1981,124,0.062595,267,374,0
4,b0266_0378,24669,1098,0.044509,266,378,0
...,...,...,...,...,...,...,...
442,b0280_0393,388,1,0.002577,280,393,9
443,b0280_0399,52637,131,0.002489,280,399,9
444,b0281_0398,106802,245,0.002294,281,398,9
445,b0281_0399,29934,66,0.002205,281,399,9


### Check the spread

Mean nearest neighbor distance between picked blocks (in block grid units) against a random
same size draw. A ratio near 1.0 means the picks are about as spread out as chance; well
below 1.0 means they are clumped. A pure global density sort at this row budget comes out
around 0.54; this should land noticeably closer to 1.0.

In [40]:
def mean_nn_dist(coords):
    tree = cKDTree(coords)
    dist, _ = tree.query(coords, k=2)
    return dist[:, 1].mean()


rng = np.random.default_rng(SEED)
all_coords = block_summary[["brow", "bcol"]].to_numpy()
picked_coords = selected_blocks[["brow", "bcol"]].to_numpy()
n_picked = len(picked_coords)

# Picks the same number of blocks we picked and compares their average distances to see how abnormal of a draw
# was made. 
random_nn = np.mean(
    [mean_nn_dist(all_coords[rng.choice(len(all_coords), n_picked, replace=False)]) for _ in range(50)]
)
picked_nn = mean_nn_dist(picked_coords)

print(f"mean nearest neighbor distance: picked {picked_nn:.2f} vs random draw {random_nn:.2f}")
print(f"ratio: {picked_nn / random_nn:.2f}")

mean nearest neighbor distance: picked 1.07 vs random draw 1.23
ratio: 0.87


## Step 6: materialize the row level pool

Second pass over the same files, full columns this time, filtered down to the selected
blocks. Filter pushdown means each file only decodes rows that match, no need to load the
embedding columns for blocks that got dropped.

In [41]:
selected_ids = selected_blocks["block_id"].tolist()
pool_parts = []
for f in JOINED_FILES:
    part = pd.read_parquet(f, filters=[("block_id", "in", selected_ids)])
    if len(part):
        pool_parts.append(part)

train_pool = pd.concat(pool_parts, ignore_index=True)

## Step 7: temporal difference features between embedding years

AlphaEarth gives one embedding vector per year (2017, 2018, 2019 here). A block that's
actively turning toward development tends to show a bigger year over year shift in embedding
space than a stable block, regardless of which particular bands move, and the raw yearly
vectors don't expose that shift directly, only the diff does. For each of the 64 bands, take
the two consecutive year deltas available: 2018 minus 2017, and 2019 minus 2018. That's 128
new columns; both source years are float32 so the diffs stay float32 too rather than doubling
to float64.

In [42]:
BAND_IDS = [f"{i:02d}" for i in range(64)]
YEAR_PAIRS = [(2018, 2017), (2019, 2018)]

# Build every diff column in a dict first and concat once at the end, rather than
# assigning columns into train_pool one at a time in the loop, which fragments the
# frame and triggers pandas's PerformanceWarning at this column count.
diff_cols = {}
for band in BAND_IDS:
    for y2, y1 in YEAR_PAIRS:
        diff_cols[f"A{band}_diff_{y2}_{y1}"] = train_pool[f"A{band}_{y2}"] - train_pool[f"A{band}_{y1}"]

train_pool = pd.concat([train_pool, pd.DataFrame(diff_cols, index=train_pool.index)], axis=1)


In [43]:
train_pool.loc[train_pool["label"] == 2, "label"] = 0 # Convert conversions to non-wetland to 0. 

train_pool = train_pool[['dist_to_developed_2019_m', 'label', 'A00_2017', 'A01_2017', 'A02_2017', 'A03_2017', 'A04_2017', 'A05_2017', 
                         'A06_2017', 'A07_2017', 'A08_2017', 'A09_2017', 'A10_2017', 'A11_2017', 'A12_2017', 'A13_2017', 'A14_2017', 
                         'A15_2017', 'A16_2017', 'A17_2017', 'A18_2017', 'A19_2017', 'A20_2017', 'A21_2017', 'A22_2017', 'A23_2017', 
                         'A24_2017', 'A25_2017', 'A26_2017', 'A27_2017', 'A28_2017', 'A29_2017', 'A30_2017', 'A31_2017', 'A32_2017', 
                         'A33_2017', 'A34_2017', 'A35_2017', 'A36_2017', 'A37_2017', 'A38_2017', 'A39_2017', 'A40_2017', 'A41_2017', 
                         'A42_2017', 'A43_2017', 'A44_2017', 'A45_2017', 'A46_2017', 'A47_2017', 'A48_2017', 'A49_2017', 'A50_2017', 
                         'A51_2017', 'A52_2017', 'A53_2017', 'A54_2017', 'A55_2017', 'A56_2017', 'A57_2017', 'A58_2017', 'A59_2017', 
                         'A60_2017', 'A61_2017', 'A62_2017', 'A63_2017', 'A00_2018', 'A01_2018', 'A02_2018', 'A03_2018', 'A04_2018', 
                         'A05_2018', 'A06_2018', 'A07_2018', 'A08_2018', 'A09_2018', 'A10_2018', 'A11_2018', 'A12_2018', 'A13_2018', 
                         'A14_2018', 'A15_2018', 'A16_2018', 'A17_2018', 'A18_2018', 'A19_2018', 'A20_2018', 'A21_2018', 'A22_2018', 
                         'A23_2018', 'A24_2018', 'A25_2018', 'A26_2018', 'A27_2018', 'A28_2018', 'A29_2018', 'A30_2018', 'A31_2018', 
                         'A32_2018', 'A33_2018', 'A34_2018', 'A35_2018', 'A36_2018', 'A37_2018', 'A38_2018', 'A39_2018', 'A40_2018', 
                         'A41_2018', 'A42_2018', 'A43_2018', 'A44_2018', 'A45_2018', 'A46_2018', 'A47_2018', 'A48_2018', 'A49_2018', 
                         'A50_2018', 'A51_2018', 'A52_2018', 'A53_2018', 'A54_2018', 'A55_2018', 'A56_2018', 'A57_2018', 'A58_2018', 
                         'A59_2018', 'A60_2018', 'A61_2018', 'A62_2018', 'A63_2018', 'A00_2019', 'A01_2019', 'A02_2019', 'A03_2019', 
                         'A04_2019', 'A05_2019', 'A06_2019', 'A07_2019', 'A08_2019', 'A09_2019', 'A10_2019', 'A11_2019', 'A12_2019', 
                         'A13_2019', 'A14_2019', 'A15_2019', 'A16_2019', 'A17_2019', 'A18_2019', 'A19_2019', 'A20_2019', 'A21_2019', 
                         'A22_2019', 'A23_2019', 'A24_2019', 'A25_2019', 'A26_2019', 'A27_2019', 'A28_2019', 'A29_2019', 'A30_2019', 
                         'A31_2019', 'A32_2019', 'A33_2019', 'A34_2019', 'A35_2019', 'A36_2019', 'A37_2019', 'A38_2019', 'A39_2019', 
                         'A40_2019', 'A41_2019', 'A42_2019', 'A43_2019', 'A44_2019', 'A45_2019', 'A46_2019', 'A47_2019', 'A48_2019', 
                         'A49_2019', 'A50_2019', 'A51_2019', 'A52_2019', 'A53_2019', 'A54_2019', 'A55_2019', 'A56_2019', 'A57_2019', 
                         'A58_2019', 'A59_2019', 'A60_2019', 'A61_2019', 'A62_2019', 'A63_2019', 'block_id', 'A00_diff_2018_2017', 
                         'A00_diff_2019_2018', 'A01_diff_2018_2017', 'A01_diff_2019_2018', 'A02_diff_2018_2017', 'A02_diff_2019_2018', 
                         'A03_diff_2018_2017', 'A03_diff_2019_2018', 'A04_diff_2018_2017', 'A04_diff_2019_2018', 'A05_diff_2018_2017', 
                         'A05_diff_2019_2018', 'A06_diff_2018_2017', 'A06_diff_2019_2018', 'A07_diff_2018_2017', 'A07_diff_2019_2018', 
                         'A08_diff_2018_2017', 'A08_diff_2019_2018', 'A09_diff_2018_2017', 'A09_diff_2019_2018', 'A10_diff_2018_2017', 
                         'A10_diff_2019_2018', 'A11_diff_2018_2017', 'A11_diff_2019_2018', 'A12_diff_2018_2017', 'A12_diff_2019_2018', 
                         'A13_diff_2018_2017', 'A13_diff_2019_2018', 'A14_diff_2018_2017', 'A14_diff_2019_2018', 'A15_diff_2018_2017', 
                         'A15_diff_2019_2018', 'A16_diff_2018_2017', 'A16_diff_2019_2018', 'A17_diff_2018_2017', 'A17_diff_2019_2018', 
                         'A18_diff_2018_2017', 'A18_diff_2019_2018', 'A19_diff_2018_2017', 'A19_diff_2019_2018', 'A20_diff_2018_2017', 
                         'A20_diff_2019_2018', 'A21_diff_2018_2017', 'A21_diff_2019_2018', 'A22_diff_2018_2017', 'A22_diff_2019_2018', 
                         'A23_diff_2018_2017', 'A23_diff_2019_2018', 'A24_diff_2018_2017', 'A24_diff_2019_2018', 'A25_diff_2018_2017', 
                         'A25_diff_2019_2018', 'A26_diff_2018_2017', 'A26_diff_2019_2018', 'A27_diff_2018_2017', 'A27_diff_2019_2018', 
                         'A28_diff_2018_2017', 'A28_diff_2019_2018', 'A29_diff_2018_2017', 'A29_diff_2019_2018', 'A30_diff_2018_2017', 
                         'A30_diff_2019_2018', 'A31_diff_2018_2017', 'A31_diff_2019_2018', 'A32_diff_2018_2017', 'A32_diff_2019_2018', 
                         'A33_diff_2018_2017', 'A33_diff_2019_2018', 'A34_diff_2018_2017', 'A34_diff_2019_2018', 'A35_diff_2018_2017', 
                         'A35_diff_2019_2018', 'A36_diff_2018_2017', 'A36_diff_2019_2018', 'A37_diff_2018_2017', 'A37_diff_2019_2018', 
                         'A38_diff_2018_2017', 'A38_diff_2019_2018', 'A39_diff_2018_2017', 'A39_diff_2019_2018', 'A40_diff_2018_2017', 
                         'A40_diff_2019_2018', 'A41_diff_2018_2017', 'A41_diff_2019_2018', 'A42_diff_2018_2017', 'A42_diff_2019_2018', 
                         'A43_diff_2018_2017', 'A43_diff_2019_2018', 'A44_diff_2018_2017', 'A44_diff_2019_2018', 'A45_diff_2018_2017', 
                         'A45_diff_2019_2018', 'A46_diff_2018_2017', 'A46_diff_2019_2018', 'A47_diff_2018_2017', 'A47_diff_2019_2018', 
                         'A48_diff_2018_2017', 'A48_diff_2019_2018', 'A49_diff_2018_2017', 'A49_diff_2019_2018', 'A50_diff_2018_2017', 
                         'A50_diff_2019_2018', 'A51_diff_2018_2017', 'A51_diff_2019_2018', 'A52_diff_2018_2017', 'A52_diff_2019_2018', 
                         'A53_diff_2018_2017', 'A53_diff_2019_2018', 'A54_diff_2018_2017', 'A54_diff_2019_2018', 'A55_diff_2018_2017', 
                         'A55_diff_2019_2018', 'A56_diff_2018_2017', 'A56_diff_2019_2018', 'A57_diff_2018_2017', 'A57_diff_2019_2018', 
                         'A58_diff_2018_2017', 'A58_diff_2019_2018', 'A59_diff_2018_2017', 'A59_diff_2019_2018', 'A60_diff_2018_2017', 
                         'A60_diff_2019_2018', 'A61_diff_2018_2017', 'A61_diff_2019_2018', 'A62_diff_2018_2017', 'A62_diff_2019_2018', 
                         'A63_diff_2018_2017', 'A63_diff_2019_2018']]

print(f"train_pool: {train_pool.shape[0]:,} rows, {train_pool.shape[1]:,} columns")
print(f"memory: {train_pool.memory_usage(deep=False).sum() / 1e9:.2f} GB")

train_pool: 10,233,887 rows, 323 columns
memory: 13.34 GB


# Save to disk. 

In [ ]:
train_pool.to_parquet("data/datasets/data_pool.parquet")

: 